<p class="lead">Sooner or later a project stops fitting in one R session: a screening library of a million compounds, a hyperparameter sweep of five hundred fits, twenty molecular-dynamics replicas of a hundred nanoseconds each. This unit is about the engineering that follows. It begins where every optimisation should begin — with measurement, because the part of the code you think is slow usually is not — and then works outwards: vectorise, parallelise on one machine with <code>future</code>, spill to disk when the data no longer fit in memory, and finally move the whole thing to the cluster as an array job or, better, as a pipeline that can be resumed after it fails at three in the morning. Two experiments in the middle deliberately make things <em>worse</em>, because the two commonest ways to waste a week on a cluster are parallelising too finely and shipping your whole workspace to every worker.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad trata el escalamiento de una tubería de laboratorio más allá del portátil. Primero se mide: <code>microbenchmark</code>, <code>Rprof</code> y la contabilidad de memoria de <code>gc()</code> sobre un cálculo real de descriptores y similitud para la biblioteca de cribado del curso (12.018 compuestos). Luego se vectoriza, se paraleliza con <code>future</code> y <code>future.apply</code> —con la diferencia entre <em>multicore</em> (bifurcación) y <em>multisession</em> (procesos nuevos)— y se mide la curva de aceleración hasta cuatro procesos, leída frente a la ley de Amdahl y la fracción serial de Karp-Flatt. Dos experimentos muestran fallos reales: una granularidad demasiado fina hace que domine el coste de despachar tareas, y una variable global grande capturada por la función se serializa hacia cada trabajador. Después se procesan datos por bloques y fuera de memoria con <code>data.table</code> y <code>arrow</code>. Se escriben y explican guiones de SLURM (sbatch, trabajos de matriz, dependencias, scratch frente a home, sacct, puntos de control), una tubería completa de <code>targets</code> para el flujo del curso, contrastada con Snakemake y Nextflow, y definiciones de Docker y Apptainer. Se cierra con una tabla de decisión sobre qué ejecutar en el portátil, la estación de trabajo, el clúster o Colab, y el coste de una hora de GPU desperdiciada.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Profile R code for time and memory before optimising it, and read a profile well enough to know what to change.</li>
<li>Vectorise what can be vectorised, parallelise embarrassingly parallel loops with <code>future.apply</code>, and measure the speed-up instead of assuming it.</li>
<li>Process data larger than memory with chunked reading, Parquet and <code>arrow</code> datasets.</li>
<li>Submit, monitor and organise jobs on a SLURM cluster: array jobs, dependencies, resource requests, scratch versus home, staging and checkpoint/restart.</li>
<li>Express a multi-step pipeline as a <code>targets</code> workflow that is re-runnable, resumable and parallel by construction, and know when Snakemake or Nextflow is the better choice.</li>
<li>Build and use containers so that results do not depend on the machine, and decide what belongs on a laptop, a workstation, a cluster or Colab.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Run every chunk in order. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","arrow","future","future.apply","microbenchmark"))</code> once before the setup chunk. All timings are measured on the machine that knits the document, so the absolute numbers will differ on yours — the <em>ratios</em> are the lesson. Up to four worker processes are started with <code>future::plan()</code> — <code>multisession</code> where the machine allows it, forked <code>multicore</code> workers otherwise, and the document prints which — and shut down at the end; nothing is submitted anywhere. No cluster is contacted: the SLURM, <code>targets</code>, Docker and Apptainer material is written out and explained, not executed, and the chunks that hold it are marked <code>eval=FALSE</code>. The whole document runs in about three minutes.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(patchwork); library(knitr)
  library(microbenchmark); library(future); library(future.apply); library(arrow)
})
set.seed(0)
PAL   <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
OKABE <- c("#0072B2", "#D55E00", "#009E73", "#E69F00", "#56B4E9", "#CC79A7")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp  <- function(f) data_file(f)
TMP <- tempdir()
peak_mb <- function() { g <- gc(); sum(g[, ncol(g)]) }      ## "max used (Mb)", both cell types
cat("R", paste(R.version$major, R.version$minor, sep = "."), "|", R.version$platform, "\n")
cat("cores: detectCores()", parallel::detectCores(), "| parallelly::availableCores()",
    parallelly::availableCores(), " (inside a container these often disagree; trust the scheduler)\n")
cat("data.table threads", getDTthreads(), "| future", as.character(packageVersion("future")),
    "| arrow", as.character(packageVersion("arrow")), "\n")
cat("supportsMulticore (fork available here):", future::supportsMulticore(), "\n")



## 1. Measure first

The oldest rule in performance engineering is that programmers are bad at guessing where time goes, and the famous warning against premature optimisation is a statement about *evidence*, not about effort<sup class="c"><a href="#r1">[1]</a></sup>. In computational chemistry the guess is wrong in a specific, repeatable way: students optimise the arithmetic and leave the string handling alone, when turning a SMILES string into something comparable costs more than every number computed from it.

Every timing quoted below was produced by the chunk above it while this page was knitted<sup class="c"><a href="#r28">[28]</a></sup>, so the numbers describe the machine that built the document and not a machine someone remembers. So the order of operations in this unit is fixed. Measure the workload. Find the dominant cost. Remove it if it can be removed (vectorisation, a better algorithm, a cheaper representation). Only then spend processes on it, because parallelism multiplies throughput but never makes a wasteful computation less wasteful — and on a shared cluster it multiplies the waste by the number of cores you asked for.

<div class="box bridge"><h4>Domain context — the laboratory's four machines</h4>
<p>The group computes on four kinds of machine, and most wasted time comes from running a job on the wrong one. A <strong>laptop</strong> runs analysis, figures and models on tables of up to a few hundred thousand rows; it is where this entire course runs. The <strong>laboratory workstation</strong> (many cores, a lot of memory, one consumer GPU) runs week-long jobs that must not be interrupted, local <strong>molecular-dynamics</strong> simulations — numerical integration of the equations of motion of a protein in water, producing a trajectory of coordinates over time — and model training that needs a GPU for hours. The <strong>university or national cluster</strong> (in Chile, the NLHPC) runs anything that is <em>many independent jobs</em>: a docking campaign over a million compounds, twenty simulation replicas, a five-hundred-point hyperparameter sweep. <strong>Colab</strong> is for teaching, for demonstrations and for a quick GPU when the workstation is busy; it is not for anything whose loss at the end of a twelve-hour session would hurt. A cluster allocation is shared, finite and audited: a job that asks for 32 cores and uses one, or a GPU job that spends its hours loading data, is charged in full and reduces the group's fair-share priority for everyone else.</p></div>

The running workload for this unit is a ligand-based screening step the laboratory repeats constantly: for every molecule in a screening library — here the frozen benchmark library of the virtual-screening unit, built from ChEMBL actives and property-matched decoys<sup class="c"><a href="#r21">[21]</a></sup> — derive a few descriptors from its SMILES string and compare it with a panel of reference actives. It is **embarrassingly parallel** — each molecule is independent of every other — which makes it the ideal case for parallelism and therefore the honest case for measuring how much parallelism really buys.

<div class="box bridge"><h4>Substitution — descriptors without RDKit</h4>
<p>The Python edition of this unit calls RDKit once per molecule (parse, seven descriptors, drug-likeness score). R has no RDKit binding in this environment, so the workload below is written in plain R: it tokenises the SMILES string, counts heavy and aromatic atoms, rings and branches, adds a crude atom-contribution logP, and computes the Jaccard (Tanimoto) similarity of the molecule's set of character 3-mers against sixteen reference actives. The physics is cruder than RDKit's, and the curated descriptor columns of the library are the ones a real analysis would use; what is faithful is the <em>shape</em> of the computation — a per-molecule, string-handling, non-vectorisable task of about half a millisecond — which is what the rest of the unit measures and scales.</p></div>



In [ ]:
library_dt <- fread(dp("screening_library.csv"))
smiles_all <- library_dt$std_smiles
ATOM_LOGP <- c(C = 0.20, c = 0.25, N = -0.35, n = -0.30, O = -0.30, o = -0.25,
               S = 0.30, s = 0.35, F = 0.20, P = 0.10, I = 0.60, B = 0.10)
PUNCT <- c("(", ")", "[", "]", "=", "#", "@", "+", "-", "/", "\\", "H", as.character(0:9))
kmer_set <- function(s, k = 3L) {
  n <- nchar(s); if (n < k) return(s)
  unique(substring(s, seq_len(n - k + 1L), seq(k, n)))
}
REFS <- lapply(library_dt[label == 1][seq_len(16), std_smiles], kmer_set)
featurise <- function(s) {
  ch     <- strsplit(s, "", fixed = TRUE)[[1]]
  heavy  <- sum(!ch %in% PUNCT)
  arom   <- sum(ch %in% c("c", "n", "o", "s", "p"))
  rings  <- length(unique(ch[ch %in% as.character(1:9)]))
  branch <- sum(ch == "(")
  logp   <- sum(ATOM_LOGP[ch], na.rm = TRUE)
  hbd    <- lengths(regmatches(s, gregexpr("OH|NH|\\[nH\\]", s)))
  km     <- kmer_set(s)
  sim    <- vapply(REFS, function(r) length(intersect(km, r)) / length(union(km, r)), numeric(1))
  c(heavy = heavy, arom = arom, rings = rings, branch = branch, logp = logp,
    hbd = hbd, max_sim = max(sim))
}
cat(sprintf("screening library: %d molecules, %d actives\n", nrow(library_dt), sum(library_dt$label)))
print(round(featurise(smiles_all[1]), 3))
mb <- microbenchmark(featurise(smiles_all[1]), times = 100L)
per_call <- median(mb$time) / 1e9
t0 <- Sys.time(); invisible(lapply(smiles_all[1:2000], featurise))
t_2000 <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
cat(sprintf("one molecule: %.3f ms (microbenchmark, 100 repeats)\n", 1e3 * per_call))
cat(sprintf("2,000 molecules: %.2f s -> %.3f ms each\n", t_2000, 1e3 * t_2000 / 2000))
for (nm in list(c(nrow(library_dt), "this library"), c(1e6, "one million compounds"),
                c(1e7, "a ten-million enumeration"))) {
  secs <- as.numeric(nm[1]) * t_2000 / 2000
  cat(sprintf("projected serial time for %-26s (%10s molecules): %8.1f min = %6.2f h\n",
              nm[2], format(as.numeric(nm[1]), big.mark = ","), secs / 60, secs / 3600))
}



Two seconds of measurement have already decided the architecture of the project. At a fraction of a millisecond per molecule the whole teaching library is a few seconds, a million compounds is a coffee break, and ten million is an hour-scale job that is worth splitting — and a real toolkit's descriptors are several times more expensive than this crude stand-in, which moves every one of those projections up by the same factor.

Note what the projection assumes: that the cost per molecule is constant. It is not exactly — large molecules have more characters and more 3-mers — but for planning, a linear extrapolation from a measured sample is the right instrument, and it is far better than the alternative of not estimating at all. The group's rule for cluster jobs is to measure on one per cent of the data, multiply, add fifty per cent, and request that as the time limit.

## 2. Where does the time actually go?

`Rprof()` is R's sampling profiler: it interrupts the interpreter at a fixed interval and records the call stack, so the cost of profiling is a few per cent rather than the two- or threefold slowdown of an instrumenting profiler. `summaryRprof()` turns the samples into two tables — time spent *in* each function (`self`) and time spent in it *and its callees* (`total`). The self table is the one that says what to change.



In [ ]:
prof_file <- file.path(TMP, "featurise.Rprof")
Rprof(prof_file, interval = 0.002, memory.profiling = TRUE)
invisible(lapply(smiles_all, featurise))
Rprof(NULL)
sr   <- summaryRprof(prof_file, memory = "both")
self <- head(sr$by.self, 8)
self$function_name <- rownames(self)
kable(self[, c("function_name", "self.time", "self.pct", "total.time", "total.pct")],
      row.names = FALSE, digits = 2,
      col.names = c("function", "self (s)", "self %", "total (s)", "total %"))
cat(sprintf("\nsampled %.2f s of execution over %d molecules at a 2 ms interval\n",
            sr$sampling.time, length(smiles_all)))


In [ ]:
pf <- head(sr$by.self, 8); pf$fun <- factor(rownames(pf), levels = rev(rownames(pf)))
print(ggplot(pf, aes(self.time, fun)) +
  geom_col(fill = OKABE[1], width = 0.65) +
  geom_text(aes(label = sprintf("%.2f s  (%.0f%%)", self.time, self.pct)), hjust = -0.1, size = 2.6) +
  xlim(0, max(pf$self.time) * 1.45) +
  labs(x = "time inside the function itself (s)", y = NULL,
       title = sprintf("Rprof: the eight most expensive calls, %s molecules",
                       format(length(smiles_all), big.mark = ","))))
cat(sprintf("the single most expensive call is %s: %.2f s (%.0f%% of the sampled time)\n",
            rownames(pf)[1], pf$self.time[1], pf$self.pct[1]))
cat(sprintf("the eight shown account for %.0f%% of the sampled time\n", sum(pf$self.pct)))



<p class="caption"><b>Figure 1.</b> The eight functions that own the time when the whole library is featurised, measured with <code>Rprof</code> at a 2 ms sampling interval and sorted by time spent in the function itself. The set operations behind the similarity calculation — <code>unique</code>, <code>union</code>, <code>intersect</code> — dominate; the descriptor counting that the code spends most of its lines on is barely visible. Any optimisation that does not touch the top bars cannot help, and the obvious one is algorithmic rather than syntactic: precompute each reference's 3-mer set once (already done), or replace the sets with a hashed bit vector.</p>

This is the result that decides everything downstream, and it is the same conclusion the Python edition reaches with RDKit: the cost is inside a per-molecule operation on a structure that is not a rectangle of numbers, so there is nothing to vectorise. The workload is therefore **parallel or nothing** — which is exactly why it is used here, and why Section 5 measures what parallelism actually delivers.

Before that, the second resource: memory.

## 3. Memory: measure it, then stop loading everything

R's memory accounting is `gc()`, which reports current and maximum use since the last reset, separately for the two kinds of internal cell. Peak memory is the number that decides whether a job survives on a node with a limit — a cluster job that exceeds its request is killed, not slowed.

The teaching library is only a few megabytes, which is too small to make the point, so the chunk below first stacks it twenty times into a realistically awkward file and then reads it two ways.



In [ ]:
big_path <- file.path(TMP, "library_big.csv")
fwrite(rbindlist(rep(list(library_dt), 20)), big_path)
invisible(gc(reset = TRUE, full = TRUE)); base_mb <- peak_mb()
invisible(gc(reset = TRUE, full = TRUE))
whole <- fread(big_path, showProgress = FALSE)
peak_full <- peak_mb() - base_mb
n_total <- nrow(whole); rm(whole); invisible(gc(reset = TRUE, full = TRUE))
hdr  <- names(fread(big_path, nrows = 0)); icol <- which(hdr == "MolWt")
CH   <- 20000L; total_mw <- 0; n_seen <- 0L
for (i in seq(0, (n_total - 1L) %/% CH)) {
  chunk <- fread(big_path, skip = 1L + i * CH, nrows = CH, select = icol,
                 header = FALSE, showProgress = FALSE)
  total_mw <- total_mw + sum(chunk[[1]]); n_seen <- n_seen + nrow(chunk)
}
peak_chunk <- peak_mb() - base_mb
cat(sprintf("stacked file: %s rows, %.0f MB on disk\n",
            format(n_total, big.mark = ","), file.size(big_path) / 1e6))
cat(sprintf("read the whole file          : peak +%.1f MB above the empty session\n", peak_full))
cat(sprintf("stream it in %s-row chunks, one column: peak +%.1f MB (%.0f times less)\n",
            format(CH, big.mark = ","), peak_chunk, peak_full / max(peak_chunk, 0.1)))
cat(sprintf("both compute the same answer: mean MolWt = %.2f Da over %s rows\n",
            total_mw / n_seen, format(n_seen, big.mark = ",")))
cat(sprintf("projected peak for a 50x larger file, read whole: %.1f GB; streamed: unchanged\n",
            50 * peak_full / 1000))



The chunked pass holds one column of twenty thousand rows at a time, so its peak is set by the chunk and not by the file: the ratio printed above is the whole lesson, and it does not change when the file grows. The cost is that the loop computes a running sum by hand — chunking trades convenience for a ceiling on memory, and the ceiling is what lets a job declare `--mem=4G` and mean it.



In [ ]:
csv_path <- big_path                                  ## the stacked 240,360-row file
pq_path  <- file.path(TMP, "library_big.parquet")
write_parquet(fread(csv_path, showProgress = FALSE), pq_path)
invisible(read_parquet(pq_path))                      ## warm the engine before timing
med <- function(f, n = 5) median(vapply(seq_len(n), function(i) system.time(f())[["elapsed"]], numeric(1)))
cols  <- c("MolWt", "MolLogP", "TPSA", "QED")
t_csv <- med(function() fread(csv_path, showProgress = FALSE))
t_pq  <- med(function() read_parquet(pq_path))
t_col <- med(function() read_parquet(pq_path, col_select = all_of(cols)))
store <- data.frame(what = c("CSV", "Parquet", "Parquet, 4 of 25 columns"),
                    MB = c(file.size(csv_path), file.size(pq_path), file.size(pq_path)) / 1e6,
                    seconds = c(t_csv, t_pq, t_col))
kable(store, digits = 3, col.names = c("stored as", "file size (MB)", "read time (s)"))
cat(sprintf("\n%s rows: Parquet is %.1f times smaller than CSV and reads %.1f times faster;\n",
            format(n_total, big.mark = ","), file.size(csv_path) / file.size(pq_path), t_csv / t_pq))
cat(sprintf("reading 4 of the 25 columns takes %.0f%% of the full Parquet read (median of 5 runs each)\n",
            100 * t_col / t_pq))


In [ ]:
g1 <- ggplot(store[1:2, ], aes(what, MB, fill = what)) + geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.1f", MB)), vjust = -0.4, size = 2.8) +
  scale_fill_manual(values = c(CSV = OKABE[2], Parquet = OKABE[3])) +
  ylim(0, max(store$MB) * 1.25) + labs(x = NULL, y = "file size (MB)", title = "Storage")
g2 <- ggplot(store[1:2, ], aes(what, seconds, fill = what)) + geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.3f s", seconds)), vjust = -0.4, size = 2.8) +
  scale_fill_manual(values = c(CSV = OKABE[2], Parquet = OKABE[3])) +
  ylim(0, max(store$seconds) * 1.3) + labs(x = NULL, y = "read time (s)", title = "Full read")
g3 <- ggplot(store[2:3, ], aes(factor(what, levels = store$what[2:3]), seconds, fill = what)) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.3f s", seconds)), vjust = -0.4, size = 2.8) +
  scale_fill_manual(values = setNames(c(OKABE[3], OKABE[1]), store$what[2:3])) +
  scale_x_discrete(labels = c("all 25", "4 columns")) +
  ylim(0, max(store$seconds[2:3]) * 1.3) + labs(x = NULL, y = "read time (s)", title = "Column pruning")
print(g1 | g2 | g3)
cat("the same 240,360-row table; each timing is the median of five reads\n")



<p class="caption"><b>Figure 2.</b> The stacked 240,360-row table stored as CSV and as Parquet, each read five times with the median reported. Parquet is columnar and compressed, so it is several times smaller on disk and reads faster; reading four of the twenty-five columns touches only those columns' data pages and is faster again. The ratios printed above the figure are the measured ones on the machine that knitted this page; the advantage grows with the file, which is why the group's docking-output tables are stored this way.</p>

Three storage rules follow and are worth applying from the first day of a project. Keep the raw download as it arrived, compressed, and never edit it. Keep working tables as Parquet, **partitioned** by the column you filter on most (`library/target=ROCK1/part-0.parquet`), so that a query reads only the relevant directories. And write CSV only at the boundary, for a supplement or a collaborator who will open it in a spreadsheet.

When a table no longer fits in memory at all, the R answer is an **Arrow dataset**: `arrow::open_dataset()` points at a file or a directory of Parquet files and returns an object that `dplyr` verbs build a query against, executed in C++ over batches without ever materialising the whole table — the same lazy, out-of-core idea as Dask in Python<sup class="c"><a href="#r6">[6]</a></sup>, and the same split-apply-combine shape that large-scale batch frameworks formalised twenty years ago<sup class="c"><a href="#r17">[17]</a></sup>. `duckdb` is the other common answer and takes SQL instead.



In [ ]:
suppressPackageStartupMessages(library(dplyr))
ds <- open_dataset(pq_path)
agg <- ds |> group_by(role) |> summarise(n = n(), mean_MolWt = mean(MolWt),
                                         mean_QED = mean(QED)) |> collect()
kable(as.data.frame(agg), digits = 3)
cat(sprintf("\nthe query above never loaded the table: the Arrow engine streamed %s rows in batches\n",
            format(nrow(library_dt), big.mark = ",")))
cat("rule of thumb: data.table up to a few GB on a workstation; Arrow or duckdb beyond that,\n")
cat("or whenever the data are already partitioned across files\n")



## 4. Vectorisation: the free speed-up, and its limit

When the computation *is* arithmetic over columns, the difference between writing it row by row and writing it as vector expressions is not a few per cent: R's arithmetic operators are compiled loops over contiguous memory, and the interpreter overhead that dominates a row loop disappears. The next chunk scores every molecule in the library with a simple desirability function — a weighted combination of drug-likeness, lipophilicity and polar surface area — in four styles that produce identical numbers.



In [ ]:
desc <- as.data.frame(library_dt[, .(QED, MolLogP, TPSA)])
score_loop <- function(df) {
  out <- numeric(nrow(df))
  for (i in seq_len(nrow(df))) {
    out[i] <- 0.4 * df$QED[i] + 0.3 * (1 - abs(df$MolLogP[i] - 3) / 5) + 0.3 * (1 - df$TPSA[i] / 200)
  }
  out
}
score_grow <- function(df) {                     ## the classic R anti-pattern: growing a vector
  out <- c()
  for (i in seq_len(nrow(df))) {
    out <- c(out, 0.4 * df$QED[i] + 0.3 * (1 - abs(df$MolLogP[i] - 3) / 5) + 0.3 * (1 - df$TPSA[i] / 200))
  }
  out
}
score_apply <- function(df) apply(df, 1, function(r)
  0.4 * r[["QED"]] + 0.3 * (1 - abs(r[["MolLogP"]] - 3) / 5) + 0.3 * (1 - r[["TPSA"]] / 200))
score_vec <- function(df)
  0.4 * df$QED + 0.3 * (1 - abs(df$MolLogP - 3) / 5) + 0.3 * (1 - df$TPSA / 200)
fns <- list(`grow a vector` = score_grow, `preallocated loop` = score_loop,
            `apply over rows` = score_apply, vectorised = score_vec)
time_it <- function(f, min_sec = 0.05) {       ## repeat until the timer is trustworthy
  reps <- 1L
  repeat {
    t <- system.time(for (i in seq_len(reps)) f())[["elapsed"]]
    if (t >= min_sec || reps > 1e4) return(t / reps)
    reps <- reps * 4L
  }
}
times <- sapply(fns, function(f) time_it(function() f(desc)))
res   <- lapply(fns, function(f) f(desc))
stopifnot(all(abs(res[["vectorised"]] - res[["preallocated loop"]]) < 1e-12),
          all(abs(res[["vectorised"]] - unname(res[["apply over rows"]])) < 1e-12))
for (nm in names(times)) {
  cat(sprintf("%-18s %8.1f ms   speed-up vs growing: %7.1fx\n",
              nm, 1e3 * times[[nm]], times[["grow a vector"]] / times[[nm]]))
}
feat_ms <- 1e3 * nrow(library_dt) * t_2000 / 2000
cat(sprintf("\nall four agree to %.1e\n", max(abs(res[["grow a vector"]] - res[["vectorised"]]))))
cat(sprintf("the featurisation of the same molecules (Section 1) costs %.0f ms - %.0f times the\n",
            feat_ms, feat_ms / (1e3 * times[["vectorised"]])))
cat("vectorised scoring, and cannot be vectorised at all\n")


In [ ]:
vdf <- data.frame(what = c(names(times), "featurisation\n(not vectorisable)"),
                  ms = c(1e3 * as.numeric(times), feat_ms))
vdf$what <- factor(vdf$what, levels = vdf$what)
print(ggplot(vdf, aes(what, ms, fill = what)) + geom_col(width = 0.6, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.1f ms", ms)), vjust = -0.4, size = 2.6) +
  scale_y_log10(limits = c(NA, max(vdf$ms) * 6)) +
  scale_fill_manual(values = c(OKABE[2], OKABE[4], OKABE[5], OKABE[3], "grey55")) +
  labs(x = NULL, y = "time for the whole library (ms, log scale)",
       title = "Vectorisation buys orders of magnitude - where it applies") +
  theme(axis.text.x = element_text(size = 6.5)))
cat("the grey bar is the parallelisation target: it is the only step that cannot be removed\n")



<p class="caption"><b>Figure 3.</b> Time to score the whole library four ways, on a logarithmic scale, with the featurisation of the same molecules for comparison. Growing a vector inside a loop is the slowest, because R reallocates and copies the whole vector at every iteration; preallocating removes that cost entirely; <code>apply</code> — which many students believe is "vectorised" — is no faster than a preallocated loop, because it still calls an R function once per row and coerces the data frame to a matrix first; the vector expression is orders of magnitude faster than all of them. The grey bar dwarfs the four: the expensive part of this pipeline is the part vectorisation cannot touch.</p>

That is the honest summary of vectorisation. It is free, it should always be done, and in a cheminformatics pipeline it usually saves a fraction of a second out of a job measured in hours. The hours are in per-molecule operations on strings and structures, and the only way to reduce those is to run them at the same time.

<div class="exercise"><h4>Exercise 1</h4>
<p>Profile a different workload: replace the 3-mer similarity in <code>featurise</code> with a comparison against the frozen Morgan fingerprints (<code>data/screening_morgan_sparse.csv.gz</code>, rebuilt with <code>Matrix::sparseMatrix</code>), computing Tanimoto as <code>c / (a + b - c)</code> with one sparse matrix product. Re-run the profile of Section 2. Which call dominates now, and does the answer change the architecture — is the job still parallel-or-nothing, or has the expensive part become a matrix product that one call to a tuned library does for all 12,018 molecules at once?</p></div>

## 5. Parallelism, honestly measured

An **embarrassingly parallel** problem is one that splits into independent pieces with no communication between them: featurising a library, docking a catalogue, running replicas of a simulation, fitting the points of a hyperparameter grid. Nearly everything the laboratory scales up is of this kind, which is fortunate, because it is the only kind that scales well without expertise.

R has no global interpreter lock, and also no threads at the user level: the unit of parallelism is a **process** — a consequence of the language's single-threaded evaluation model<sup class="c"><a href="#r26">[26]</a></sup> and the reason the Python edition's "threads made it slower" experiment has no counterpart here. The `future` framework<sup class="c"><a href="#r22">[22]</a></sup> gives all of them one interface — you choose a *plan* and the same code runs under it — and `future.apply` provides drop-in replacements for the `apply` family. Three plans matter:

- `plan(sequential)` — no parallelism; the baseline every measurement needs.
- `plan(multicore, workers = n)` — **fork**: the worker is a copy of the current session and inherits everything in memory, so starting it is almost free. It is unavailable on Windows and unsafe inside RStudio and inside some knitting front-ends, which is why `future::supportsMulticore()` is printed in the setup chunk.
- `plan(multisession, workers = n)` — a fresh R session per worker, started by `callr`-style mechanics: safe everywhere, but each worker must be started (about a second) and every object the function needs must be *serialised into it*.

The chunk below asks for `multisession` first and falls back to `multicore` if the environment refuses it — the machine that knits this document runs inside a sandbox with no listening sockets, so the PSOCK workers a `multisession` plan needs cannot be started, and the measured curve is a fork curve. The chunk prints which backend it used, and the difference matters for the reading: fork inherits memory copy-on-write and therefore pays almost nothing for worker start-up or for data transfer, so a fork curve is the *optimistic* case. Under `multisession` — what a Windows laptop and most RStudio sessions get — the same code pays the serialisation measured in Section 6, and the curve is flatter.



In [ ]:
shards <- function(x, n) split(x, rep(seq_len(n), length.out = length(x)))
featurise_chunk <- function(chunk) lapply(chunk, featurise)
MS_OK <- tryCatch({                            ## probe once: PSOCK workers need a local socket
  cl <- parallelly::makeClusterPSOCK(1L, autoStop = TRUE); parallel::stopCluster(cl); TRUE
}, error = function(e) FALSE)
use_plan <- function(n) {                      ## multisession where it works, fork otherwise
  if (MS_OK) { plan(multisession, workers = n); "multisession (new sessions)" }
  else       { plan(multicore,   workers = n); "multicore (fork)" }
}
BACKEND <- NA_character_
elapsed <- numeric(0)
plan(sequential)
t0 <- Sys.time(); invisible(featurise_chunk(smiles_all))
elapsed["1"] <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
startup <- numeric(0)
for (n in c(2, 3, 4)) {
  t0 <- Sys.time(); BACKEND <- use_plan(n)
  invisible(future_lapply(seq_len(n), function(i) i, future.seed = TRUE))   ## warm the workers
  startup[as.character(n)] <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
  t0 <- Sys.time()
  invisible(future_lapply(shards(smiles_all, n), featurise_chunk, future.seed = TRUE))
  elapsed[as.character(n)] <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
}
plan(sequential)
speedup <- elapsed[["1"]] / elapsed
cat("backend used for the parallel points:", BACKEND, "\n")
for (n in names(elapsed)) {
  cat(sprintf("%s worker(s): %5.2f s   speed-up %4.2fx%s\n", n, elapsed[[n]], speedup[[n]],
              if (n == "1") "" else sprintf("   (worker start-up %.2f s, excluded)", startup[[n]])))
}



Now the arithmetic that turns those seconds into a decision. **Amdahl's law** says that if a fraction \( p \) of a program is parallelisable and the rest is irreducibly serial, the speed-up on \( N \) workers is bounded by

\[ S(N) = \frac{1}{(1-p) + p/N}. \]

Work it by hand for the plausible case \( p = 0.95 \) and \( N = 4 \): the denominator is \( 0.05 + 0.2375 = 0.2875 \), so \( S = 3.48 \). With \( N = 16 \) the denominator is \( 0.05 + 0.059 = 0.109 \) and \( S = 9.2 \); with infinitely many workers it is \( 1/0.05 = 20 \). Five per cent of serial code caps the whole enterprise at twentyfold, no matter how large the allocation — which is why Gustafson's reply, that in practice one enlarges the problem rather than shortening the run, matters so much for science<sup class="c"><a href="#r9">[9]</a><a href="#r10">[10]</a></sup>.

Running the law backwards gives the **Karp–Flatt metric**: from a measured speed-up \( S \) on \( N \) workers, the experimentally determined serial fraction is

\[ e = \frac{1/S - 1/N}{1 - 1/N}. \]



In [ ]:
workers <- as.integer(names(elapsed))
karp    <- (1 / speedup[-1] - 1 / workers[-1]) / (1 - 1 / workers[-1])
p_fit   <- 1 - mean(karp)
curve   <- data.frame(n = workers, measured = as.numeric(speedup),
                      amdahl = 1 / ((1 - p_fit) + p_fit / workers), ideal = workers)
g1 <- ggplot(curve, aes(n)) +
  geom_line(aes(y = ideal), linetype = 2, linewidth = 0.4) +
  geom_line(aes(y = amdahl), colour = OKABE[4], linewidth = 0.7) +
  geom_line(aes(y = measured), colour = OKABE[1], linewidth = 0.7) +
  geom_point(aes(y = measured), colour = OKABE[1], size = 1.8) +
  geom_text(aes(y = measured, label = sprintf("%.2fx", measured)), vjust = 1.8, size = 2.5) +
  scale_x_continuous(breaks = workers) +
  labs(x = "worker processes", y = "speed-up",
       title = sprintf("multisession parallelism (dashed: ideal; orange: Amdahl, p = %.3f)", p_fit)) +
  theme(plot.title = element_text(size = 8))
g2 <- ggplot(data.frame(n = workers[-1], e = as.numeric(karp)), aes(n, e)) +
  geom_line(colour = OKABE[2], linewidth = 0.7) + geom_point(colour = OKABE[2], size = 1.8) +
  scale_x_continuous(breaks = workers[-1]) + ylim(0, max(0.5, max(karp) * 1.3)) +
  labs(x = "worker processes", y = "Karp-Flatt serial fraction e",
       title = "Serial fraction implied by each point")
print(g1 | g2)
cat(sprintf("measured speed-up on %d workers: %.2fx (%.0f%% of linear)\n",
            max(workers), speedup[[length(speedup)]], 100 * speedup[[length(speedup)]] / max(workers)))
cat(sprintf("Karp-Flatt serial fraction per point: %s\n", paste(round(karp, 3), collapse = ", ")))
cat(sprintf("hand-worked Amdahl prediction for p = 0.95 on 4 workers: %.2fx\n", 1 / (0.05 + 0.95 / 4)))
cat(sprintf("with the fitted p = %.3f, Amdahl predicts %.1fx on 16 workers and %.0fx in the limit\n",
            p_fit, 1 / ((1 - p_fit) + p_fit / 16), 1 / (1 - p_fit)))



<p class="caption"><b>Figure 4.</b> Measured speed-up of the featurisation against worker processes, with the ideal linear line and the Amdahl curve implied by the fitted serial fraction. Four processes recover appreciably less than the fourfold maximum; with the forking backend used here the gap is the collection of results back from the workers and the fact that the shards are not perfectly balanced, and under a <code>multisession</code> plan the serialisation of the inputs into each worker would be added to it. Worker start-up is measured separately and excluded from the timings, which is generous to the parallel version — in a real job it is paid too. The right-hand panel shows the serial fraction implied by each individual point, and its <em>direction</em> is the diagnostic: a fraction that falls as workers are added — the case here — means a fixed cost being amortised over more work, while one that rises means a per-worker overhead that will keep growing and caps the useful request.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Using the fitted serial fraction, compute by hand the speed-up you should expect on 32 cores of the cluster, and the number of cores beyond which each additional core buys less than five per cent. Then decide what to request in the <code>--cpus-per-task</code> line of the SLURM script in Section 7 — remembering that a larger request waits longer in the queue.</p></div>

## 6. Two ways to make parallelism slower than serial

### Tasks that are too small

Parallelism has a fixed cost per task: the arguments are serialised, sent to the worker, deserialised, and the result makes the return journey. When a task takes a microsecond, that overhead is thousands of times the work. `future.apply` chunks automatically for exactly this reason; the experiment below defeats the chunking with `future.scheduling = Inf` (one future per element) to show what the chunking is protecting you from.



In [ ]:
trivial <- smiles_all[1:500]
t_serial <- time_it(function() lapply(trivial, nchar))     ## too fast to time once
invisible(use_plan(4))
invisible(future_lapply(1:4, function(i) i, future.seed = TRUE))
t_fine <- system.time(invisible(future_lapply(trivial, nchar, future.seed = TRUE,
                                              future.scheduling = Inf)))[["elapsed"]]
t_chunked <- system.time(invisible(future_lapply(trivial, nchar, future.seed = TRUE)))[["elapsed"]]
plan(sequential)
gran <- data.frame(how = c("serial lapply", "4 workers, one future per element",
                           "4 workers, default chunking"),
                   seconds = c(t_serial, t_fine, t_chunked))
kable(gran, digits = 4)
cat(sprintf("\nthe serial loop takes %.2f ms; one future per element is %s times slower, and the default\n",
            1e3 * t_serial, format(round(t_fine / t_serial), big.mark = ",")))
cat(sprintf("chunking is still %s times slower than not parallelising at all\n",
            format(round(t_chunked / t_serial), big.mark = ",")))
cat(sprintf("the featurisation task costs about %.2f ms per molecule, far above the per-task overhead -\n",
            1e3 * t_2000 / 2000))
cat("which is why it parallelises and this one does not\n")


In [ ]:
gran$how <- factor(gran$how, levels = gran$how)
print(ggplot(gran, aes(how, seconds, fill = how)) + geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.0f ms", 1e3 * seconds)), vjust = -0.4, size = 2.7) +
  scale_y_log10(limits = c(NA, max(gran$seconds) * 8)) +
  scale_fill_manual(values = c(OKABE[3], OKABE[2], OKABE[4])) +
  labs(x = NULL, y = "time for 500 trivial tasks (s, log scale)",
       title = "When the task is smaller than the cost of dispatching it") +
  theme(axis.text.x = element_text(size = 6.5)))
cat("the rule is quantitative: a task must be large compared with a millisecond of dispatch\n")
cat("overhead, and when it is not, the fix is bigger tasks rather than more workers\n")



<p class="caption"><b>Figure 5.</b> Five hundred trivial tasks — counting the characters of a string — run serially, dispatched one future per element to four workers, and dispatched with <code>future.apply</code>'s default chunking. One future per element is orders of magnitude slower than not parallelising at all; the default chunking recovers almost all of the loss but still does not beat the serial loop, because there is nothing to gain. The measured numbers are printed above the figure.</p>

### Globals that travel to every worker

The second failure mode is specific to R and catches everyone once. A future captures the **globals** its expression refers to and sends a copy to each worker. A function that mentions a large object in the calling environment therefore serialises that object into every worker, even if it only reads one number from it. The chunk below measures the serialisation cost directly — that is the quantity a `multisession` plan pays, per worker — and then times forty tasks written both ways.



In [ ]:
BIG <- matrix(runif(3e6), ncol = 30)                      ## about 24 MB
uses_big   <- function(i) c(i, nrow(BIG))                 ## mentions BIG: it is a global of the future
uses_small <- local({ nr <- nrow(BIG); function(i) c(i, nr) })   ## captures one number instead
size_mb <- as.numeric(object.size(BIG)) / 1e6
t_ser   <- system.time(invisible(serialize(BIG, NULL)))[["elapsed"]]
invisible(use_plan(4))
t_big   <- system.time(invisible(future_lapply(1:40, uses_big, future.seed = TRUE)))[["elapsed"]]
t_small <- system.time(invisible(future_lapply(1:40, uses_small, future.seed = TRUE)))[["elapsed"]]
plan(sequential)
cat(sprintf("BIG is %.0f MB; serialising it once costs %.2f s\n", size_mb, t_ser))
cat(sprintf("active backend: %s\n", BACKEND))
cat(sprintf("40 tasks that mention BIG  : %.2f s\n", t_big))
cat(sprintf("40 tasks that capture a number: %.2f s\n", t_small))
cat(sprintf("under a multisession plan the first version would pay about %.2f s per worker start,\n",
            t_ser))
cat(sprintf("that is %.1f s for 4 workers, before any work is done; a forking plan pays none of it\n",
            4 * t_ser))
rm(BIG); invisible(gc())



Three further process-parallel pitfalls complete the list, all of which the group has hit.

**Oversubscription.** `data.table` and the BLAS behind matrix algebra are already multi-threaded. Four worker processes that each start a multi-threaded `data.table` produce far more threads than cores, and everything slows down. Inside workers, call `data.table::setDTthreads(1)` and set `OMP_NUM_THREADS=1`, and let the outer level own the parallelism.

**Memory multiplication.** Every worker holds its own copy of what it receives. Four workers each receiving a two-gigabyte table need eight gigabytes, and the cluster kills the job. Pass file paths and row ranges, not data — which is also what makes the array job of Section 7 work.

Read the two timings against the backend the chunk printed. Under `multisession` the version that mentions the matrix is several times slower, and the gap is the serialisation. Under `multicore` — the backend available here — the child process is a copy-on-write fork, so the matrix is never copied and the two timings coincide: the pitfall is real but invisible on a forking platform, which is exactly why code developed on a Linux workstation falls over when a colleague runs it on Windows. The portable habit is to capture what you need (`local({ nr <- nrow(BIG); function(i) ... })`) rather than the object that contains it, and to check `future::futureAssign`'s reported globals size when a parallel loop is mysteriously slow.

**Random numbers.** Independent workers must not draw the same stream. `future.apply` handles this when you pass `future.seed = TRUE` (as every call above does), which uses L'Ecuyer-CMRG streams; forgetting it is a silent correctness bug, not a performance one.
## 7. The cluster: SLURM in practice

Beyond one machine, work is submitted to a **scheduler**, which queues jobs, allocates nodes and accounts for usage. SLURM is the one almost every academic cluster runs, including the Chilean national facility<sup class="c"><a href="#r2">[2]</a></sup>. The mental model is short: you write a shell script with resource requests at the top, submit it with `sbatch`, and the scheduler runs it when the resources are free. Everything else is detail — but the details are where allocations are wasted.



In [ ]:
## featurise.slurm - written and explained here; nothing is submitted from this document.
##   #!/bin/bash
##   #SBATCH --job-name=featurise          # appears in squeue; make it searchable
##   #SBATCH --partition=general           # which queue; ask the cluster documentation
##   #SBATCH --nodes=1                     # one machine: this job does not use MPI
##   #SBATCH --ntasks=1                    # one process launched by SLURM...
##   #SBATCH --cpus-per-task=16            # ...which itself uses 16 cores (our future workers)
##   #SBATCH --mem=32G                     # hard limit: exceeding it kills the job
##   #SBATCH --time=04:00:00               # hard limit: exceeding it kills the job
##   #SBATCH --output=logs/%x_%j.out       # %x job name, %j job id - never overwrite a log
##   #SBATCH --error=logs/%x_%j.err
##   #SBATCH --mail-type=END,FAIL
##   set -euo pipefail                     # fail fast, fail loudly
##
##   module load R/4.5.0                   # environment modules: the cluster's software catalogue
##
##   SCRATCH=/scratch/$USER/$SLURM_JOB_ID  # fast, local, temporary: work here
##   mkdir -p "$SCRATCH"
##   trap 'rm -rf "$SCRATCH"' EXIT         # always clean up, even on failure
##
##   cp "$HOME/data/library.parquet" "$SCRATCH/"         # stage input in
##   export OMP_NUM_THREADS=1                            # avoid oversubscription in the workers
##   Rscript --vanilla featurise.R \
##       --input   "$SCRATCH/library.parquet" \
##       --output  "$SCRATCH/features.parquet" \
##       --workers "$SLURM_CPUS_PER_TASK"                # ask SLURM how many cores you were given
##   cp "$SCRATCH/features.parquet" "$HOME/results/"     # stage results out


In [ ]:
R_SIDE <- c(
 'args    <- commandArgs(trailingOnly = TRUE)',
 'workers <- as.integer(Sys.getenv("SLURM_CPUS_PER_TASK", unset = "1"))',
 'future::plan(future::multicore, workers = workers)   ## fork: cheap on a Linux node',
 'data.table::setDTthreads(1)                          ## the outer level owns the parallelism',
 'cat(sprintf("job %s on %s: %d workers\\n", Sys.getenv("SLURM_JOB_ID"),',
 '            Sys.info()[["nodename"]], workers))')
cat(R_SIDE, sep = "\n")
cat("\n\nsubmit with:  sbatch featurise.slurm      inspect with:  squeue -u $USER\n")
cat("after it ends: sacct -j <jobid> --format=JobID,JobName,Elapsed,AllocCPUS,MaxRSS,State,ExitCode\n")
cat(sprintf("\nthis document measured %.2f s for %s molecules on 4 workers; at that rate 16 cores\n",
            elapsed[["4"]], format(length(smiles_all), big.mark = ",")))
cat(sprintf("would need about %.0f min for ten million molecules, so --time=04:00:00 is generous\n",
            1e7 * (elapsed[["4"]] / length(smiles_all)) * (4 / 16) / 60))



Five of those lines deserve a sentence each, because they are the ones students get wrong.

`--cpus-per-task` and `$SLURM_CPUS_PER_TASK`. Request what you will use and *read the request back* in the program, as the R side above does. Hard-coding `workers = 32` in a job that was given four cores produces a job that runs slowly and annoys everyone else on the node; asking for 32 and using one wastes 31.

`--mem` and `--time`. Both are hard limits enforced by the kernel and the scheduler: exceeding memory kills the job immediately, exceeding the walltime kills it at the limit — usually just before the results are written. Measure on a sample, multiply, add a margin, and check afterwards with `sacct` what you actually used.

**Scratch versus home.** The home filesystem is small, backed up and shared; the scratch filesystem is large, fast, local to the cluster and periodically deleted. Jobs read and write on scratch and copy results home at the end. Running an I/O-heavy job directly on a network home directory is the most effective way to slow a cluster down for everyone.

**Staging.** Copy inputs in and results out explicitly. If the job dies, the `trap` line still removes the scratch directory; results that were never copied are lost, which is the argument for checkpointing.

**Logs with `%j`.** A log file whose name contains the job id is a log you can still read next week. A log called `out.txt` is overwritten by the next job.

### Array jobs: one script, a thousand pieces

The natural way to run an embarrassingly parallel campaign on a cluster is not a bigger machine but an **array job**: one script submitted once, run many times with a different index each time. The scheduler packs the tasks onto whatever nodes are free, and a failed task can be resubmitted alone.



In [ ]:
## dock_array.slurm
##   #!/bin/bash
##   #SBATCH --job-name=dock_array
##   #SBATCH --array=0-99%20               # 100 tasks, at most 20 running at once
##   #SBATCH --cpus-per-task=4
##   #SBATCH --mem=8G
##   #SBATCH --time=02:00:00
##   #SBATCH --output=logs/%x_%A_%a.out    # %A array job id, %a task index
##   set -euo pipefail
##   module load R/4.5.0
##
##   SHARD=$(printf "%03d" "$SLURM_ARRAY_TASK_ID")
##   IN=$HOME/data/shards/library_$SHARD.parquet
##   OUT=$HOME/results/features_$SHARD.parquet
##
##   if [[ -f "$OUT" ]]; then              # idempotence: a resubmitted task does no work twice
##       echo "shard $SHARD already done"; exit 0
##   fi
##   Rscript --vanilla featurise.R --input "$IN" --output "$OUT.tmp" \
##           --workers "$SLURM_CPUS_PER_TASK"
##   mv "$OUT.tmp" "$OUT"                  # atomic rename: a partial file never looks finished
##
## merge.slurm
##   #!/bin/bash
##   #SBATCH --job-name=merge --cpus-per-task=2 --mem=16G --time=00:30:00
##   set -euo pipefail
##   Rscript --vanilla merge.R --inputs "$HOME/results/features_*.parquet" \
##                             --output "$HOME/results/features_all.parquet"


In [ ]:
cat("submit the array, then a merge job that starts only if the whole array succeeded:\n")
cat("  jid=$(sbatch --parsable dock_array.slurm)\n")
cat("  sbatch --dependency=afterok:$jid merge.slurm\n")
cat("re-run only the tasks that failed:\n")
cat("  sbatch --array=7,23,88 dock_array.slurm\n\n")
n_tasks <- 100; per_task_min <- 45; concurrent <- 20
cat(sprintf("%d tasks of %d min, %d at a time -> about %.1f h wall-clock for %d core-hours of work\n",
            n_tasks, per_task_min, concurrent, n_tasks * per_task_min / concurrent / 60,
            round(n_tasks * per_task_min / 60)))


In [ ]:
set.seed(0)
durations <- pmin(pmax(rnorm(100, 45, 8), 20), 80)      ## minutes, simulated for the picture
slots <- numeric(20); starts <- numeric(100); slot_of <- integer(100)
for (i in seq_along(durations)) {
  s <- which.min(slots); starts[i] <- slots[s]; slot_of[i] <- s; slots[s] <- slots[s] + durations[i]
}
gantt <- data.frame(task = seq_along(durations), slot = slot_of,
                    start = starts / 60, end = (starts + durations) / 60)
print(ggplot(gantt, aes(xmin = start, xmax = end, ymin = slot - 0.4, ymax = slot + 0.4, fill = task)) +
  geom_rect(colour = "white", linewidth = 0.2) +
  scale_fill_viridis_c(guide = "none") +
  geom_vline(xintercept = max(slots) / 60, colour = OKABE[2], linetype = 2, linewidth = 0.5) +
  annotate("text", x = max(slots) / 60 + 0.1, y = 3, hjust = 0, size = 2.6, colour = OKABE[2],
           label = sprintf("last task ends\nat %.1f h", max(slots) / 60)) +
  labs(x = "hours since submission", y = "concurrent slot",
       title = "A simulated 100-task array job with --array=0-99%20"))
cat(sprintf("total work %.1f core-hours; wall-clock %.1f h on %d slots; slot efficiency %.0f%%\n",
            sum(durations) / 60, max(slots) / 60, length(slots),
            100 * sum(durations) / (max(slots) * length(slots))))
cat("the unused tail is load imbalance: the last slots finish early and wait for the slowest task\n")



<p class="caption"><b>Figure 6.</b> A simulated 100-task array job restricted to twenty concurrent slots, each bar a task with a duration drawn from a normal distribution around 45 minutes. The total work is the sum of the bars; the wall-clock time is the length of the longest slot, and the slot efficiency printed above is how much of the reserved capacity the campaign actually used. The ragged right edge is load imbalance — the reason to shuffle a library before sharding it, so that no shard is full of the largest molecules — and the dashed line is the moment the dependent merge job is allowed to start.</p>

### Checkpoint and restart

A simulation that runs for three days and dies on day two with nothing written has consumed three days of allocation and produced nothing. Long jobs therefore write **checkpoints**: molecular-dynamics engines write a restart file every few thousand steps, a hyperparameter sweep writes each completed fit to its own `.rds`, a featurisation writes one output per shard. Three rules make checkpointing work: write to a temporary name and rename atomically (as the array script does), make every task idempotent so that resubmitting it is harmless, and record enough state to resume rather than to restart. With those in place, a job that hits its walltime can be requeued with `--dependency=afternotok` and finish the remainder.

## 8. Workflow managers: the pipeline as a graph of targets

A pipeline written as a shell script runs from the beginning every time and knows nothing about what has already been done. A **workflow manager** inverts the description: you declare what each output depends on and what command produces it, and the manager works out the rest — which steps are out of date, which can run at the same time, and how to submit them to the cluster. Re-running after a failure repeats only the missing steps, which is the property that makes a long pipeline survivable<sup class="c"><a href="#r3">[3]</a><a href="#r4">[4]</a><a href="#r8">[8]</a></sup>.

In R the native answer is **`targets`**<sup class="c"><a href="#r23">[23]</a></sup>, a Make-like, function-oriented toolkit: the pipeline is a list of targets, each defined by an R expression, and the manager hashes the code and the upstream values to decide what is out of date. It stores results in `_targets/objects/` rather than in files you name, which is both its convenience and the thing to understand before using it. `targets` is **not installed in this environment**, so the pipeline below is written out as non-evaluated source and explained; nothing is run.



In [ ]:
## _targets.R - curation -> sharding -> featurisation -> model -> evaluation -> report
library(targets)
library(tarchetypes)                       ## tar_map, tar_render and other patterns

tar_option_set(
  packages  = c("data.table", "ranger", "rockcurate"),
  format    = "qs",                        ## fast serialisation for large objects
  storage   = "worker", retrieval = "worker",
  resources = tar_resources(crew = tar_resources_crew(controller = "cluster"))
)

controller <- crew.cluster::crew_controller_slurm(   ## one SLURM job per worker
  name = "cluster", workers = 20, seconds_idle = 60,
  script_lines = "module load R/4.5.0",
  slurm_memory_gigabytes_per_cpu = 4, slurm_cpus_per_task = 4)
tar_option_set(controller = controller)

N_SHARDS <- 8L
SPLITS   <- c("random", "scaffold")

list(
  tar_target(raw_file, "data/rock1_chembl_raw.csv", format = "file"),   ## tracked by hash
  tar_target(keys_file, "data/rock1_chembl_curated.csv", format = "file"),
  tar_target(curated, rockcurate::curate(rockcurate::read_raw(raw_file),
                                         rockcurate::structure_keys(fread(keys_file)))),
  tar_target(shard_id, seq_len(N_SHARDS)),
  tar_target(shard, split_library(curated$table, N_SHARDS)[[shard_id]],
             pattern = map(shard_id)),                                  ## 8 dynamic branches
  tar_target(features, featurise_shard(shard), pattern = map(shard)),   ## 8 parallel jobs
  tar_target(feature_table, rbindlist(features)),
  tar_target(split_name, SPLITS),
  tar_target(model, fit_model(feature_table, split = split_name, seed = 0),
             pattern = map(split_name)),                                ## 2 branches
  tar_target(metrics, evaluate_model(model, feature_table, split = split_name),
             pattern = map(model, split_name)),
  tarchetypes::tar_render(report, "report.Rmd")                         ## depends on metrics
)



Four features of that file carry the whole idea.

**Targets are declarations, not steps.** Nothing says "first curate, then featurise". `targets` reads the expressions, sees that `features` mentions `shard` and that `shard` mentions `curated`, and derives the order; asking for `report` pulls the entire graph.

**Branching creates jobs.** `pattern = map(shard_id)` turns one target into eight, and `map(split_name)` turns the model into two. The list of shards appears once.

**Resources are declared per pipeline (and can be per target).** A `crew` controller turns each worker into a SLURM job with its own `--cpus-per-task` and memory, so the same pipeline runs locally with `tar_make()` and on the cluster with `tar_make()` plus the controller — no script rewriting.

**Invalidation is by hash, not by timestamp.** If you edit the body of `fit_model`, `targets` notices that the *code* of the target changed and reruns it and everything downstream; if you only reformat it, nothing reruns. File targets (`format = "file"`) are hashed too, which is why `raw_file` is declared that way.

The operational commands are short: `tar_manifest()` lists the targets, `tar_outdated()` says what would run, `tar_make()` runs it, `tar_visnetwork()` draws the graph with the up-to-date targets in green, and `tar_meta()` holds the provenance — the time, the seed and the hash of every target, which is the record a paper's supplementary methods should quote.



In [ ]:
nodes <- data.frame(
  name = c("raw_file", "curated", "shard\n1..8", "features\n1..8", "feature_table",
           "model\nrandom", "model\nscaffold", "metrics\nrandom", "metrics\nscaffold", "report"),
  x = c(0.50, 0.50, 0.50, 0.50, 0.50, 0.28, 0.72, 0.28, 0.72, 0.50),
  y = c(0.96, 0.82, 0.68, 0.54, 0.40, 0.28, 0.28, 0.16, 0.16, 0.03),
  done = c(TRUE, TRUE, TRUE, TRUE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE))
edges <- data.frame(
  from = c("raw_file", "curated", "shard\n1..8", "features\n1..8", "feature_table",
           "feature_table", "model\nrandom", "model\nscaffold", "metrics\nrandom", "metrics\nscaffold"),
  to   = c("curated", "shard\n1..8", "features\n1..8", "feature_table", "model\nrandom",
           "model\nscaffold", "metrics\nrandom", "metrics\nscaffold", "report", "report"))
e <- merge(merge(edges, nodes, by.x = "from", by.y = "name"), nodes, by.x = "to", by.y = "name",
           suffixes = c("_f", "_t"))
print(ggplot() +
  geom_segment(data = e, aes(x = x_f, y = y_f - 0.035, xend = x_t, yend = y_t + 0.045),
               arrow = arrow(length = unit(0.12, "cm"), type = "closed"),
               colour = "grey65", linewidth = 0.35) +
  geom_rect(data = nodes, aes(xmin = x - 0.095, xmax = x + 0.095, ymin = y - 0.035, ymax = y + 0.045,
                              fill = done, colour = done), linewidth = 0.5) +
  geom_text(data = nodes, aes(x, y + 0.005, label = name), size = 2.5) +
  scale_fill_manual(values = c(`TRUE` = "#dff0e4", `FALSE` = "#fdeadf"), guide = "none") +
  scale_colour_manual(values = c(`TRUE` = OKABE[3], `FALSE` = OKABE[2]), guide = "none") +
  annotate("text", x = 0.03, y = 0.95, hjust = 0, size = 2.6,
           label = "green: up to date (skipped)\norange: targets tar_make() would rebuild") +
  coord_cartesian(xlim = c(0, 1), ylim = c(-0.02, 1)) + theme_void())
cat(sprintf("targets in the graph: %d (the eight shard and feature branches are drawn as one box each)\n",
            nrow(nodes)))
cat("after a crash during model fitting, tar_make() rebuilds only the orange targets\n")



<p class="caption"><b>Figure 7.</b> The target graph of the pipeline, in the state that follows a failure during model fitting: the file target, the curation, the eight shards and the eight featurisation branches are up to date and are skipped, and only the five downstream targets are rebuilt. This is resumability, and it is a property of the declared <em>dependencies</em> rather than of any bookkeeping the user does. The eight featurisation branches depend on nothing but their own shard, so the manager runs as many of them at once as the controller's worker budget allows.</p>

**Snakemake** and **Nextflow** solve the same problem from outside R. Snakemake's unit is a *file*: rules declare input and output patterns, wildcards expand one rule into many jobs, and `conda:` directives give each rule its own environment<sup class="c"><a href="#r3">[3]</a><a href="#r5">[5]</a></sup>. Nextflow's unit is a *process* connected by typed channels, each task runs in its own directory, and containers are first class<sup class="c"><a href="#r4">[4]</a></sup>. The practical division in the group is: `targets` when the pipeline is R from end to end and the intermediate objects are R objects; Snakemake when the steps are command-line programs producing files, and the shards are naturally named by wildcards; Nextflow when the pipeline is a chain of third-party tools, must run on several clouds, or already exists as an nf-core community pipeline. All three are vastly better than a shell script, and the comparison literature agrees that the choice between them matters much less than the choice to use one at all<sup class="c"><a href="#r7">[7]</a><a href="#r8">[8]</a></sup>. Toil occupies the same niche for workflows that must span batch systems and clouds<sup class="c"><a href="#r16">[16]</a></sup>, and the provenance record any of them emits is the practical form of the "automate the analysis" rule that reproducibility guidance has repeated for a decade<sup class="c"><a href="#r24">[24]</a><a href="#r25">[25]</a></sup>.

## 9. Containers: the environment as an artefact

An `renv.lock` file reproduces the R packages, and a conda or Bioconda specification does the same for a stack that mixes R with compiled command-line tools<sup class="c"><a href="#r27">[27]</a></sup>. A **container** reproduces the operating system, the compilers, the system libraries and the R packages, as a single immutable image with a digest. It is what makes a pipeline run identically on a laptop, on a cluster node and on a reviewer's machine three years later<sup class="c"><a href="#r11">[11]</a><a href="#r12">[12]</a></sup>.

Two formats matter here. **Docker** is the ecosystem everything is built in and requires root, which clusters do not grant. **Apptainer** (formerly Singularity) runs unprivileged, uses a single-file image, and mounts the user's home and scratch directories by default — which is why it is what runs on HPC<sup class="c"><a href="#r12">[12]</a></sup>. The usual path is to build with Docker and convert. For R the base images come from the Rocker project, which pins the R version in the image tag.



In [ ]:
## Dockerfile - written and explained; not built here.
##   FROM rocker/r-ver:4.5.0                    # the R version is part of the image, not a surprise
##
##   RUN apt-get update && apt-get install -y --no-install-recommends \
##           libxml2-dev libssl-dev libcurl4-openssl-dev zlib1g-dev \
##       && rm -rf /var/lib/apt/lists/*
##
##   WORKDIR /opt/rockscale
##   COPY renv.lock renv.lock                   # dependencies first: this layer is cached
##   RUN install2.r --error renv \
##       && R -q -e "renv::restore(lockfile = 'renv.lock')"   # install2.r ships with rocker
##
##   COPY DESCRIPTION NAMESPACE ./              # then the package itself
##   COPY R ./R
##   COPY inst ./inst
##   RUN R CMD INSTALL --no-multiarch .
##
##   LABEL org.opencontainers.image.source="https://github.com/<organisation>/rockscale" \
##         org.opencontainers.image.version="1.0.0" \
##         org.opencontainers.image.licenses="MIT"
##
##   USER 1000:1000                             # never root at run time
##   WORKDIR /work                              # bind-mount the data here
##   ENTRYPOINT ["Rscript", "--vanilla"]
##   CMD ["-e", "cat(as.character(packageVersion('rockcurate')), '\n')"]
##
## rockscale.def - Apptainer
##   Bootstrap: docker
##   From: ghcr.io/<organisation>/rockscale@sha256:3f8c...   # pulled by digest, never by tag
##
##   %labels
##       Version 1.0.0
##       Maintainer LBQC, Universidad Catolica del Maule
##
##   %environment
##       export OMP_NUM_THREADS=1
##       export R_LIBS_USER=/usr/local/lib/R/site-library
##
##   %runscript
##       exec Rscript --vanilla "$@"
##
##   %help
##       Curation and featurisation pipeline. Run:
##         apptainer run --bind /scratch/$USER:/work rockscale_1.0.0.sif featurise.R --help


In [ ]:
cat("build and convert:\n")
cat("  docker build -t rockscale:1.0.0 .\n")
cat("  apptainer build rockscale_1.0.0.sif docker-daemon://rockscale:1.0.0\n")
cat("  apptainer run --bind /scratch/$USER:/work rockscale_1.0.0.sif featurise.R --help\n\n")
cat("inside a SLURM script the container replaces the module load line:\n")
cat("  apptainer exec --bind /scratch/$USER:/work $HOME/img/rockscale_1.0.0.sif \\\n")
cat("      Rscript --vanilla featurise.R --input /work/shard.parquet --output /work/out.parquet\n\n")
cat("pin by digest, not by tag: a tag can be moved, sha256:... cannot\n")



Three habits make containers worth their cost<sup class="c"><a href="#r13">[13]</a></sup>. **Order the layers by how often they change** — system packages, then R dependencies, then your source — so that editing a function does not rebuild the whole image. **Never run as root inside the container**, and never bake data or credentials into an image that will be pushed to a registry. And **refer to images by digest** in a workflow: `rockscale@sha256:…` is reproducible, `rockscale:latest` is a promise nobody keeps.

For a cluster, the environment modules (`module load`) remain the fastest path for compiled simulation engines that the centre has tuned for its hardware; containers are for everything else, and for anything you would otherwise have to ask an administrator to install.

## 10. What belongs where, and the cost of getting it wrong

The last decision is the cheapest to get right and the most expensive to get wrong: which machine a job belongs on.



In [ ]:
placement <- data.frame(
  workload = c("Exploratory analysis, figures, teaching", "Curation of a ChEMBL target",
               "Featurising 10^6 compounds", "Docking a 10^6-compound library",
               "MD: 3 replicas x 1 us", "Hyperparameter sweep, 500 fits",
               "Training a small neural model", "Fine-tuning a large pretrained model",
               "Demonstration or teaching with a GPU"),
  scale = c("< 10 min", "minutes", "hours x cores", "10^4 core-hours", "days on GPU",
            "hours x cores", "< 1 h on GPU", "hours on GPU", "minutes"),
  where = c("laptop", "laptop", "laptop or cluster (array)", "cluster (array job)",
            "workstation or cluster GPU", "cluster or workstation", "workstation GPU or Colab",
            "cluster GPU", "Colab"),
  why = c("immediate feedback matters", "runs in one process; keep it reproducible",
          "measure first: it may be minutes", "one shard per task",
          "checkpoint every few ns", "one fit per task", "fits in a session",
          "needs persistence and memory", "nothing is lost if it disappears"))
kable(placement, col.names = c("workload", "scale", "where it belongs", "why"))


In [ ]:
GPU_W <- 300; PUE <- 1.3; CPU_W <- 12; CI_CHILE <- 400       ## W, overhead, W, gCO2e per kWh
CLOUD_GPU <- 3.0; CLOUD_CPU <- 0.05                          ## USD per hour, order of magnitude
cost <- data.frame(
  scenario = c("one wasted GPU-hour", "a failed 12 h GPU job",
               "a sweep rerun because the seed was not fixed", "a 500-core-hour array job run twice"),
  hours = c(1, 12, 48, 500), kind = c("gpu", "gpu", "gpu", "cpu"))
cost$kWh    <- round(cost$hours * ifelse(cost$kind == "gpu", GPU_W, CPU_W) * PUE / 1000, 1)
cost$kgCO2e <- round(cost$kWh * CI_CHILE / 1000, 1)
cost$USD    <- round(cost$hours * ifelse(cost$kind == "gpu", CLOUD_GPU, CLOUD_CPU))
cost$scenario <- factor(cost$scenario, levels = rev(cost$scenario))
g1 <- ggplot(cost, aes(kgCO2e, scenario)) + geom_col(fill = OKABE[2], width = 0.6) +
  geom_text(aes(label = kgCO2e), hjust = -0.2, size = 2.6) + xlim(0, max(cost$kgCO2e) * 1.3) +
  labs(x = "kg CO2e (grid at 400 g/kWh)", y = NULL, title = "Carbon") +
  theme(axis.text.y = element_text(size = 6))
g2 <- ggplot(cost, aes(USD, scenario)) + geom_col(fill = OKABE[1], width = 0.6) +
  geom_text(aes(label = USD), hjust = -0.2, size = 2.6) + xlim(0, max(cost$USD) * 1.3) +
  labs(x = "equivalent on-demand cloud cost (USD)", y = NULL, title = "Money") +
  theme(axis.text.y = element_blank())
print(g1 | g2)
kable(cost[, c("scenario", "hours", "kWh", "kgCO2e", "USD")], row.names = FALSE)
cat(sprintf("\nassumptions: GPU %d W, PUE %.1f, CPU core %d W, grid %d gCO2e/kWh, cloud $%.2f/GPU-h\n",
            GPU_W, PUE, CPU_W, CI_CHILE, CLOUD_GPU))



<p class="caption"><b>Figure 8.</b> The cost of four common kinds of waste, converted into carbon and into equivalent cloud spending with the assumptions printed beneath. The estimates follow the Green Algorithms accounting — runtime times power times a data-centre overhead factor, times the carbon intensity of the local grid<sup class="c"><a href="#r14">[14]</a><a href="#r15">[15]</a></sup> — and they are deliberately coarse; the same accounting applied to large model training is what made the energy cost of deep learning a published concern<sup class="c"><a href="#r18">[18]</a></sup>. What is not coarse is the ranking: a failed long GPU job costs more than every CPU job the group runs in a month, and the fix is free — fix the seed, write checkpoints, and test the pipeline on one shard before submitting a thousand.</p>

None of this is specific to clusters: it is the "good enough practices" discipline of writing down what you did, automating it and making it restartable<sup class="c"><a href="#r19">[19]</a><a href="#r20">[20]</a></sup>, applied where mistakes are expensive. The group's checklist before any large submission is five lines: run the pipeline with `tar_outdated()` (or `snakemake -n`) and read the plan; run one shard end to end and look at its output; check that the resource request matches what one shard actually used (`sacct --format=MaxRSS,Elapsed`); make every task idempotent and atomic; and submit with a concurrency limit so that a mistake costs twenty tasks rather than a thousand.

<div class="box bridge"><h4>Bridge</h4>
<p>The pipeline being scaled here is the one built in <a href="../U6_Data_Quality/index.html">Unit 6</a> and packaged in <a href="../U24_Software_Engineering/index.html">Unit 24</a> — the <code>Rscript</code> entry point the SLURM scripts call is the one written there, and the reproducibility practices behind containers and lockfiles are <a href="../U2_Reproducibility/index.html">Unit 2</a>. The screening library profiled in Section 1 is the benchmark of the virtual-screening unit, <a href="../U19_Virtual_Screening/index.html">U19</a>; the replica simulations used as the array-job example are the data of <a href="../U20_MD_Analytics/index.html">U20</a>; the hyperparameter sweeps that motivate array jobs are <a href="../U14_Ensembles_Tuning/index.html">Unit 14</a>; the results communicated from a cluster run are reported following <a href="../U23_Communication/index.html">Unit 23</a>; and the deployment of the resulting model, which reuses the container written here, is <a href="../U26_Apps_Deployment/index.html">Unit 26</a>. In the sister courses, <a href="../../../BGM324/05_Molecular_Dynamics/index.html">Molecular dynamics</a> explains what the long GPU jobs actually compute and <a href="../../../BGM324/03_Molecular_Docking/index.html">Molecular docking</a> what a screening campaign does with a shard, while <a href="../../../Cheminformatics/U6_Databases/index.html">Databases</a> covers where the million compounds come from in the first place. The Python edition of this unit is <a href="../../Python/U25_Scaling_HPC/index.html">here</a>.</p></div>

## 11. Summary

**Measure, then optimise.** A two-second profile showed that the set operations behind the similarity calculation own the time in this workload, that the descriptor counting is irrelevant, and that nothing in the hot path can be vectorised<sup class="c"><a href="#r1">[1]</a></sup>. Every later decision followed from that.

**Vectorise what is arithmetic.** Vector expressions beat a growing-vector loop by orders of magnitude on the same calculation, preallocation removes most of the gap on its own, and `apply` is not vectorisation. But in this pipeline the vectorisable part was a rounding error next to the per-molecule string work.

**Parallelise what is independent, and measure the speed-up.** Four `multisession` workers gave clearly less than fourfold; Amdahl's law and the Karp–Flatt serial fraction turn that measurement into a prediction for the cluster<sup class="c"><a href="#r9">[9]</a><a href="#r10">[10]</a></sup>.

**Two failure modes, both exhibited.** Dispatching one future per trivial task was orders of magnitude slower than a serial loop, because the overhead per task dwarfed the task; and a function that merely *mentions* a 24 MB matrix makes that matrix a global of the future, whose serialisation cost was measured directly and is paid once per worker under a `multisession` plan (and avoided by a forking one). Chunk the work, keep the globals small, and ask what `future` has to serialise.

**Spill to disk before you run out of memory.** A chunked pass over a stacked 60 MB file used a small fraction of the memory of a full read and gave the same answer; Parquet with column pruning is the format for working tables, and `arrow::open_dataset()` the tool when a table no longer fits<sup class="c"><a href="#r6">[6]</a></sup>.

**On the cluster, think in jobs, not in machines.** An array job with a concurrency limit, atomic outputs, idempotent tasks, scratch for I/O, staged inputs, logs named by job id, and a dependent merge job is the shape of nearly every campaign the group runs<sup class="c"><a href="#r2">[2]</a></sup>.

**Let a workflow manager own the graph.** `targets` derives the order from the declared dependencies, reruns only what its hashes show to be out of date, branches over shards and splits, and submits workers to SLURM through a controller<sup class="c"><a href="#r23">[23]</a></sup>; Snakemake is the better fit for file-based pipelines of command-line tools<sup class="c"><a href="#r3">[3]</a><a href="#r5">[5]</a></sup> and Nextflow for multi-cloud chains of third-party programs<sup class="c"><a href="#r4">[4]</a></sup>.

**Containers make the environment an artefact**, pinned by digest, built with Docker from a Rocker base, run unprivileged with Apptainer on HPC<sup class="c"><a href="#r11">[11]</a><a href="#r12">[12]</a><a href="#r13">[13]</a></sup>.

**Put the job on the right machine.** The cheapest optimisation available to a student is not to run the job twice.

## 12. Self-assessment

<details><summary>1. You are about to parallelise a function. What do you measure first, and why?</summary>
<p>The serial cost per item and the profile. The cost per item decides whether the task is big enough to be worth dispatching (milliseconds, yes; microseconds, no) and lets you project the total runtime for the real data. The profile tells you whether the time is in code you could remove instead — if most of it is in an operation that a vector expression or a better algorithm could do, that is free and parallelism is not.</p></details>

<details><summary>2. Why does R not have the "threads made it slower" failure mode that Python has, and what is the R equivalent?</summary>
<p>Because R offers no user-level threads at all: <code>future</code>'s parallel plans are processes, so there is no global interpreter lock to serialise them. The R equivalents are the two shown in Section 6 — dispatch overhead when tasks are too small, and the serialisation of captured globals into every worker — plus oversubscription, where multi-threaded <code>data.table</code> or BLAS inside several worker processes produces more threads than cores.</p></details>

<details><summary>3. What is the difference between <code>multicore</code> and <code>multisession</code>, and when must you use the second?</summary>
<p><code>multicore</code> forks the current session: the worker inherits every object already in memory, so it starts in milliseconds and needs no serialisation. <code>multisession</code> starts fresh R processes and serialises into them whatever the future needs. Forking is unavailable on Windows and unsafe inside RStudio and some GUI front-ends (shared connections and graphics devices do not survive it), so portable code — including this document — uses <code>multisession</code> and pays the start-up and serialisation cost, which Section 5 measures separately.</p></details>

<details><summary>4. With a measured serial fraction of 0.1, what speed-up can you expect on 16 cores, and what on 64?</summary>
<p>Amdahl gives 1/(0.1 + 0.9/16) = 6.4 on 16 cores and 1/(0.1 + 0.9/64) = 8.8 on 64. Quadrupling the cores bought 37 % more speed. The practical conclusion is to request 16, not 64: the extra cores spend longer in the queue, are charged in full, and deliver almost nothing.</p></details>

<details><summary>5. A colleague's array job asks for 32 cores per task and uses one. What are the three consequences?</summary>
<p>The job waits far longer in the queue, because 32 free cores are rarer than one; the allocation is charged for 32 cores times the walltime, so the group's budget drains 32 times faster; and the fair-share priority of everyone in the group falls, so the next job queues longer too. <code>sacct --format=JobID,Elapsed,AllocCPUS,MaxRSS</code> after the fact is how the mismatch is discovered.</p></details>

<details><summary>6. Why does the array script write to <code>$OUT.tmp</code> and then rename?</summary>
<p>Because a rename within a filesystem is atomic: the output file either does not exist or is complete. If the task is killed halfway — walltime, memory, node failure — the partial file is a <code>.tmp</code> that nobody mistakes for a result, and the <code>if [[ -f "$OUT" ]]</code> guard at the top makes resubmitting the task harmless. Together they make the array idempotent, which is what allows <code>--array=7,23,88</code> to repair a campaign.</p></details>

<details><summary>7. What does a workflow manager give you that a shell script does not?</summary>
<p>A dependency graph, which yields three things a script cannot: re-running only what is out of date (resumability after a crash), automatic parallelism over independent targets, and a provenance record of what produced which object with which code and seed. It also separates the resource declaration from the submission mechanism, so the same pipeline runs on a laptop and on SLURM.</p></details>

<details><summary>8. In the <code>targets</code> pipeline, which targets run at the same time, and what limits them?</summary>
<p>The eight featurisation branches, because each depends only on its own shard; and, later, the two model branches, one per split. The limit is the number of workers the <code>crew</code> controller is allowed to start — each of which is itself a SLURM job with its own cores — intersected with what the queue grants. Nothing in the pipeline file changes between running locally and running on the cluster.</p></details>

<details><summary>9. Why build a container with Docker but run it with Apptainer on the cluster?</summary>
<p>Docker is where the tooling, base images and registries live, but its daemon requires root, which no cluster grants to users. Apptainer runs unprivileged, stores the image as a single file the user owns, and mounts home and scratch by default, so it fits the HPC security model. Converting a Docker image to a <code>.sif</code> is one command, so the ecosystem and the execution model can be chosen independently.</p></details>

<details><summary>10. Estimate the cost of forgetting to fix a random seed in a 12-hour GPU experiment that must therefore be repeated.</summary>
<p>Twelve GPU-hours: roughly 4-5 kWh once the data-centre overhead is included, of the order of two kilograms of CO2e on a grid at 400 g/kWh, and tens of dollars at on-demand cloud prices — plus twelve hours of queue and of your own time. The arithmetic is in Section 10, and the preventive measure costs one line of code: <code>set.seed()</code>, or <code>future.seed = TRUE</code> when the work is parallel.</p></details>

<details><summary>11. When is an Arrow dataset the right answer, and when is it a complication?</summary>
<p>It is right when a single table genuinely exceeds memory, when the data are already partitioned across many Parquet files, or when only a few columns of a wide table are needed. It is a complication when a chunked <code>fread</code> loop would do, because the lazy query must be expressible in the verbs Arrow can push down — anything else triggers a <code>collect()</code> that loads everything anyway, which is the failure students meet first. Measure the chunked version before reaching for it.</p></details>

<details><summary>12. Size two campaigns over the same million compounds: featurising them, and docking them. Which belongs on the cluster?</summary>
<p>Featurisation first, with the rate measured in Section 1 (a fraction of a millisecond per molecule in this crude R implementation, a few milliseconds with a real toolkit): 10<sup>6</sup> molecules is of the order of ten minutes of serial work, or 0.2 core-hours. Four workers on a laptop finish it in minutes, so it does not belong on the cluster at all — the queue wait alone would exceed the job.</p>
<p>Docking the same library is a different problem: at an optimistic 30 s per molecule it is 3 x 10<sup>7</sup> s, about 8,300 core-hours, which no single machine will do. Shard the shuffled library into 1,000 pieces of 1,000 compounds; each piece is about 8.3 core-hours, so with <code>--cpus-per-task=4</code> a task runs for roughly 2 h. Submit <code>--array=0-999%50 --cpus-per-task=4 --mem=8G --time=04:00:00</code>: fifty concurrent slots means twenty waves of about 2 h each, roughly two days of wall-clock time. Write each shard atomically to scratch and copy results home, add a merge job with <code>--dependency=afterok</code>, and run one shard interactively before submitting the array. The lesson is the comparison: the decision to use a cluster follows from the measured cost per item, not from the number of items.</p></details>

## Glossary

<dl class="gloss">
<dt>Amdahl's law</dt><dd>The bound S(N) = 1/((1-p) + p/N) on the speed-up achievable with N workers when a fraction p of the work is parallelisable.</dd>
<dt>Apptainer (Singularity)</dt><dd>Container runtime designed for HPC: unprivileged execution, single-file images, user directories mounted by default.</dd>
<dt>Array job</dt><dd>One SLURM submission that runs the same script many times with a different task index, optionally with a concurrency limit.</dd>
<dt>Arrow dataset</dt><dd>A lazily queried collection of Parquet files that <code>dplyr</code> verbs filter, select and aggregate in C++ without loading the whole table.</dd>
<dt>Checkpoint/restart</dt><dd>Writing enough intermediate state that a killed job resumes rather than restarts; atomic renames and idempotent tasks make it safe.</dd>
<dt>Chunking</dt><dd>Processing a file in pieces (<code>fread(skip=, nrows=)</code>) so that peak memory is set by the chunk, not by the file.</dd>
<dt>Embarrassingly parallel</dt><dd>A workload that splits into independent tasks with no communication; the only kind that scales well without expertise.</dd>
<dt>future / plan</dt><dd>R's unified interface to parallel back-ends: the same code runs under <code>sequential</code>, <code>multicore</code> (fork) or <code>multisession</code> (new sessions).</dd>
<dt>Globals (in a future)</dt><dd>The objects a parallel expression refers to, which are detected automatically and serialised into every worker — the reason a large captured object is expensive.</dd>
<dt>Karp-Flatt metric</dt><dd>The serial fraction e = (1/S - 1/N)/(1 - 1/N) inferred from a measured speed-up; its growth with N indicates overhead rather than serial code.</dd>
<dt>Oversubscription</dt><dd>Starting more threads or processes than there are cores, typically by combining an outer worker pool with multi-threaded <code>data.table</code> or BLAS.</dd>
<dt>Parquet</dt><dd>A compressed columnar file format that supports reading a subset of columns and partitioning across directories.</dd>
<dt>Profiling</dt><dd>Measuring where a program spends time (<code>Rprof</code>, <code>microbenchmark</code>) or memory (<code>gc</code>, <code>Rprof(memory.profiling = TRUE)</code>) before changing it.</dd>
<dt>Scratch filesystem</dt><dd>Fast, large, temporary cluster storage for job input and output, as opposed to the small, backed-up home filesystem.</dd>
<dt>SLURM</dt><dd>The scheduler most academic clusters run: <code>sbatch</code> to submit, <code>squeue</code> to monitor, <code>sacct</code> to audit what a finished job used.</dd>
<dt>targets</dt><dd>R's Make-like pipeline toolkit: targets declared as expressions, invalidated by hashing code and dependencies, branched with <code>pattern = map(...)</code>.</dd>
<dt>Workflow manager</dt><dd>A tool that derives execution order from declared dependencies, re-runs only what is out of date and submits jobs to a scheduler.</dd>
</dl>



In [ ]:
produced <- sort(basename(list.files(TMP, pattern = "library|featurise")))
cat("files written by this document:", paste(produced, collapse = ", "), "\n")
invisible(file.remove(list.files(TMP, pattern = "library|featurise", full.names = TRUE)))
plan(sequential)
cat("worker processes shut down; workers now available to future:", nbrOfWorkers(), "\n")
cat("scratch files removed:", length(list.files(TMP, pattern = "library|featurise")) == 0, "\n")



## References and further reading

<ol class="refs">
<li id="r1">Knuth DE. Structured programming with go to statements. <em>ACM Comput Surv</em>. 1974;6(4):261-301. <a href="https://doi.org/10.1145/356635.356640">doi:10.1145/356635.356640</a></li>
<li id="r2">Yoo AB, Jette MA, Grondona M. SLURM: simple Linux utility for resource management. In: <em>Job Scheduling Strategies for Parallel Processing</em>. LNCS 2862. Springer; 2003:44-60. <a href="https://doi.org/10.1007/10968987_3">doi:10.1007/10968987_3</a></li>
<li id="r3">Molder F, Jablonski KP, Letcher B, et al. Sustainable data analysis with Snakemake. <em>F1000Research</em>. 2021;10:33. <a href="https://doi.org/10.12688/f1000research.29032.2">doi:10.12688/f1000research.29032.2</a></li>
<li id="r4">Di Tommaso P, Chatzou M, Floden EW, Barja PP, Palumbo E, Notredame C. Nextflow enables reproducible computational workflows. <em>Nat Biotechnol</em>. 2017;35:316-319. <a href="https://doi.org/10.1038/nbt.3820">doi:10.1038/nbt.3820</a></li>
<li id="r5">Koster J, Rahmann S. Snakemake - a scalable bioinformatics workflow engine. <em>Bioinformatics</em>. 2012;28(19):2520-2522. <a href="https://doi.org/10.1093/bioinformatics/bts480">doi:10.1093/bioinformatics/bts480</a></li>
<li id="r6">Rocklin M. Dask: parallel computation with blocked algorithms and task scheduling. In: <em>Proc. 14th Python in Science Conf.</em>; 2015:126-132. <a href="https://doi.org/10.25080/Majora-7b98e3ed-013">doi:10.25080/Majora-7b98e3ed-013</a></li>
<li id="r7">Wratten L, Wilm A, Goke J. Reproducible, scalable, and shareable analysis pipelines with bioinformatics workflow managers. <em>Nat Methods</em>. 2021;18:1161-1168. <a href="https://doi.org/10.1038/s41592-021-01254-9">doi:10.1038/s41592-021-01254-9</a></li>
<li id="r8">Leipzig J. A review of bioinformatic pipeline frameworks. <em>Brief Bioinform</em>. 2017;18(3):530-536. <a href="https://doi.org/10.1093/bib/bbw020">doi:10.1093/bib/bbw020</a></li>
<li id="r9">Amdahl GM. Validity of the single processor approach to achieving large scale computing capabilities. In: <em>Proc. AFIPS Spring Joint Computer Conf.</em>; 1967:483-485. <a href="https://doi.org/10.1145/1465482.1465560">doi:10.1145/1465482.1465560</a></li>
<li id="r10">Gustafson JL. Reevaluating Amdahl's law. <em>Commun ACM</em>. 1988;31(5):532-533. <a href="https://doi.org/10.1145/42411.42415">doi:10.1145/42411.42415</a></li>
<li id="r11">Boettiger C. An introduction to Docker for reproducible research. <em>ACM SIGOPS Oper Syst Rev</em>. 2015;49(1):71-79. <a href="https://doi.org/10.1145/2723872.2723882">doi:10.1145/2723872.2723882</a></li>
<li id="r12">Kurtzer GM, Sochat V, Bauer MW. Singularity: scientific containers for mobility of compute. <em>PLoS ONE</em>. 2017;12(5):e0177459. <a href="https://doi.org/10.1371/journal.pone.0177459">doi:10.1371/journal.pone.0177459</a></li>
<li id="r13">Nust D, Sochat V, Marwick B, et al. Ten simple rules for writing Dockerfiles for reproducible data science. <em>PLoS Comput Biol</em>. 2020;16(11):e1008316. <a href="https://doi.org/10.1371/journal.pcbi.1008316">doi:10.1371/journal.pcbi.1008316</a></li>
<li id="r14">Lannelongue L, Grealey J, Inouye M. Green algorithms: quantifying the carbon footprint of computation. <em>Adv Sci</em>. 2021;8(12):2100707. <a href="https://doi.org/10.1002/advs.202100707">doi:10.1002/advs.202100707</a></li>
<li id="r15">Lannelongue L, Aronson HG, Bateman A, et al. GREENER principles for environmentally sustainable computational science. <em>Nat Comput Sci</em>. 2023;3:514-521. <a href="https://doi.org/10.1038/s43588-023-00461-y">doi:10.1038/s43588-023-00461-y</a></li>
<li id="r16">Vivian J, Rao AA, Nothaft FA, et al. Toil enables reproducible, open source, big biomedical data analyses. <em>Nat Biotechnol</em>. 2017;35:314-316. <a href="https://doi.org/10.1038/nbt.3772">doi:10.1038/nbt.3772</a></li>
<li id="r17">Dean J, Ghemawat S. MapReduce: simplified data processing on large clusters. <em>Commun ACM</em>. 2008;51(1):107-113. <a href="https://doi.org/10.1145/1327452.1327492">doi:10.1145/1327452.1327492</a></li>
<li id="r18">Strubell E, Ganesh A, McCallum A. Energy and policy considerations for deep learning in NLP. In: <em>Proc. 57th Annual Meeting of the ACL</em>; 2019:3645-3650. <a href="https://doi.org/10.18653/v1/P19-1355">doi:10.18653/v1/P19-1355</a></li>
<li id="r19">Wilson G, Bryan J, Cranston K, Kitzes J, Nederbragt L, Teal TK. Good enough practices in scientific computing. <em>PLoS Comput Biol</em>. 2017;13(6):e1005510. <a href="https://doi.org/10.1371/journal.pcbi.1005510">doi:10.1371/journal.pcbi.1005510</a></li>
<li id="r20">Taschuk M, Wilson G. Ten simple rules for making research software more robust. <em>PLoS Comput Biol</em>. 2017;13(4):e1005412. <a href="https://doi.org/10.1371/journal.pcbi.1005412">doi:10.1371/journal.pcbi.1005412</a></li>
<li id="r21">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL database in 2023. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r22">Bengtsson H. A unifying framework for parallel and distributed processing in R using futures. <em>The R Journal</em>. 2021;13(2):208-227. <a href="https://doi.org/10.32614/RJ-2021-048">doi:10.32614/RJ-2021-048</a></li>
<li id="r23">Landau WM. The targets R package: a dynamic Make-like function-oriented pipeline toolkit for reproducibility and high-performance computing. <em>J Open Source Softw</em>. 2021;6(57):2959. <a href="https://doi.org/10.21105/joss.02959">doi:10.21105/joss.02959</a></li>
<li id="r24">Sandve GK, Nekrutenko A, Taylor J, Hovig E. Ten simple rules for reproducible computational research. <em>PLoS Comput Biol</em>. 2013;9(10):e1003285. <a href="https://doi.org/10.1371/journal.pcbi.1003285">doi:10.1371/journal.pcbi.1003285</a></li>
<li id="r25">Beaulieu-Jones BK, Greene CS. Reproducibility of computational workflows is automated using continuous analysis. <em>Nat Biotechnol</em>. 2017;35:342-346. <a href="https://doi.org/10.1038/nbt.3780">doi:10.1038/nbt.3780</a></li>
<li id="r26">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r27">Gruning B, Dale R, Sjodin A, et al. Bioconda: sustainable and comprehensive software distribution for the life sciences. <em>Nat Methods</em>. 2018;15:475-476. <a href="https://doi.org/10.1038/s41592-018-0046-7">doi:10.1038/s41592-018-0046-7</a></li>
<li id="r28">Xie Y. <em>Dynamic Documents with R and knitr</em>. 2nd ed. Chapman and Hall/CRC; 2015. <a href="https://doi.org/10.1201/9781315382487">doi:10.1201/9781315382487</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Wickham H. <em>Advanced R</em>. 2nd ed. Chapman and Hall/CRC; 2019 (free online). - Chapters on performance, profiling and memory explain every measurement in Sections 1 to 4.</li>
<li>Peng RD. <em>R Programming for Data Science</em> (free online). - The profiling chapter, at an introductory level.</li>
<li>Gorelick M, Ozsvald I. <em>High Performance Python</em>. 2nd ed. O'Reilly; 2020. - Written for Python, but the cost model of processes, serialisation and task granularity is language-independent.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>SLURM documentation, especially the pages on <code>sbatch</code>, job arrays and <code>sacct</code>: <a href="https://slurm.schedmd.com/">slurm.schedmd.com</a>.</li>
<li>NLHPC (Laboratorio Nacional de Computacion de Alto Rendimiento, Chile) user documentation: <a href="https://www.nlhpc.cl/">nlhpc.cl</a>.</li>
<li><code>future</code> and <code>future.apply</code> documentation, including the vignette on common issues with globals: <a href="https://future.futureverse.org/">future.futureverse.org</a>.</li>
<li>The <code>targets</code> user manual and its HPC chapter: <a href="https://books.ropensci.org/targets/">books.ropensci.org/targets</a>; Snakemake: <a href="https://snakemake.readthedocs.io/">snakemake.readthedocs.io</a>; nf-core: <a href="https://nf-co.re/">nf-co.re</a>.</li>
<li>Arrow for R, "Working with Arrow datasets": <a href="https://arrow.apache.org/docs/r/">arrow.apache.org/docs/r</a>.</li>
<li>Rocker project images and the Apptainer user guide: <a href="https://rocker-project.org/">rocker-project.org</a>, <a href="https://apptainer.org/docs/">apptainer.org/docs</a>.</li>
<li>Green Algorithms calculator for the carbon footprint of a computation: <a href="https://www.green-algorithms.org/">green-algorithms.org</a>.</li>
</ul>
